## 3.2张量：多维数组

### 3.2.2 构造第一个张量

In [4]:
import torch
a = torch.ones(3)
a

tensor([1., 1., 1.])

In [5]:
a[1]

tensor(1.)

In [6]:
float(a[1])

1.0

In [7]:
a[2]

tensor(1.)

### 3.2.3张量的本质

PyTorch张量或者NumPy数组通常是连续内存块的视图...

使用一维张量，将x坐标存储在偶数索引中，将y坐标存储在奇数索引中。

In [41]:
points = torch.zeros(6)
points[0] = 4.0
points[1] = 1.0
points[2] = 5.0
points[3] = 3.0
points[4] = 2.0
points[5] = 1.0

我们还可以通过向构造函数传入一个python列表来达到同样的效果：

In [9]:
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0])
points

tensor([4., 1., 5., 3., 2., 1.])

为了得到第1个点的坐标，我们执行以下操作：

In [10]:
float(points[0]), float(points[1])

(4.0, 1.0)

我们可以用二维张量

In [11]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

这里我们构造了一个元素为列表的列表，我们可以通过下面的操作查看张量的**形状**：

In [12]:
points.shape

torch.Size([3, 2])

通过以上的操作我们知道了每个维度上张量的大小。
我们还可以使用zeros()或ones()函数来初始化张量，以元组的形式来指定大小。

In [13]:
points = torch.zeros(3, 2)
points

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

现在我们可以通过2个索引来访问张量中的元素

In [14]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [15]:
points[0, 1]

tensor(1.)

这将返回数据集中第一个点的y坐标。我们也可以像之前那样访问张量中的第一个元素，得到第一个点的二维坐标。

In [16]:
points[0]

tensor([4., 1.])

输出是另一个张量，它**展示了相同基础数据的不同视图**。

## 3.3索引张量

In [17]:
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

如果我们需要得到一个张量中除了第1点以外的所有点？

使用范围索引表示法很容易实现。

In [18]:
some_list = list(range(6))
some_list[:]  # 所有元素
some_list[1:4]  # 从第1个元素（包含）待第4个元素（不包含）
some_list[1:]  # 从第1个元素（包含）到末尾
some_list[:4]  # 从列表开始到第4个元素（不包含）
some_list[:-1]  # 从列表开始到最后一个元素之前的元素
some_list[1:4:2]  # 从第1个元素（包含）到第4个元素（不包含），移动步长为2

[1, 3]

我们可以对torch使用同样的表示法。

In [19]:
points[1:]  # 代表第1行之后的所有行

tensor([[5., 3.],
        [2., 1.]])

In [20]:
points[1:, :]  # 第一行后的所有行、所有列

tensor([[5., 3.],
        [2., 1.]])

In [21]:
points[1:, 0]  # 第1行后的所有行的第一列

tensor([5., 2.])

In [22]:
points[None].shape  # 增加大小为1的维度，就像unsqueeze()一样

torch.Size([1, 3, 2])

## 3.4 命名张量

在通过过个张量转换数据时，跟踪哪个维度包含哪些数据容易出错。
为了简单，我们将使用虚拟数据，并将其转换为灰度图像。

In [23]:
img_t = torch.randn(3, 5, 5)
weights = torch.tensor([0.2126, 0.7152, 0.0722])
batch_t = torch.randn(2, 3, 5, 5)

惰性的未加权平均值可以写成下面的形式：

In [24]:
img_gray_naive = img_t.mean(-3)
batch_gray_naive = batch_t.mean(-3)
img_gray_naive.shape, batch_gray_naive.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

PyTorch还允许我们对形状相同的张量进行乘法运算（不是矩阵乘法，只是相乘），也允许与给定维度中 其中一个操作数为1的张量进行运算。它还会自动附加大小为1的**前导维度**，这个特性被称为**广播**。

形状为（2， 3， 5， 5）的batch_t乘以一个形状为（3， 1， 1）的unsqueezed_weights张量，**可以得到一个形状为（2,3,5,5）的张量**。

In [25]:
# 在张量的最后一个维度（-1 代表最后一个维度）之后，再增加一个维度；
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze_(-1)
# torch.unsqueeze_(-1) 或 tensor.unsqueeze_(-1)，是 unsqueeze 的就地（in-place）版本， 它会直接修改原始张量，为其增加一个维度，并且返回修改后的原始张量。（此处非必要，可以都使用非就地的方式）
img_weights = (img_t * unsqueezed_weights)
batch_weights = (batch_t * unsqueezed_weights)
img_gray_weights = img_weights.sum(-3)
batch_gray_weights = batch_weights.sum(-3)
batch_weights.shape, batch_t.shape, unsqueezed_weights.shape

(torch.Size([2, 3, 5, 5]), torch.Size([2, 3, 5, 5]), torch.Size([3, 1, 1]))

因为这会使程序很快变得混乱。为了提高效率，PyTorch根据NumPy改编的einsum()函数指定了一种**微型语言索引**，为这些乘积的总和的维度提供索引名。

在python中，广播（一种用来概括未命名事物的形式）通常使用三个点（...）来表示。

In [26]:
img_gray_weighted_fancy = torch.einsum('...chw,c->...hw', img_t, weights)
batch_gray_weighted_fancy = torch.einsum('...chw,c->...hw', batch_t, weights)
batch_gray_weighted_fancy.shape

torch.Size([2, 5, 5])

PyTorch1.3将命名张量作为实验性的特性。**张量工厂函数**（诸如tensor()和rand()函数）有一个**name**参数，该参数是一个字符串序列。

In [27]:
weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names=['channels'])
weights_named

/var/folders/bt/6vdf5z9109vb7n7fplq48kf80000gn/T/ipykernel_13715/2371314847.py:1: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/c10/core/TensorImpl.h:1938.)
  weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names=['channels'])


tensor([0.2126, 0.7152, 0.0722], names=('channels',))

当已经有一个张量并且想为其添加名称**但不修改当前的名称**，可以对其调用**refine_names()方法**，与索引类似，省略号（...）允许省略任意数量的维度。

使用rename()方法，还可以覆盖和删除（通过传入None）现有名称。

In [28]:
img_named = img_t.refine_names(..., 'channels', 'rows', 'columns')
batch_named = batch_t.refine_names(..., 'channels', 'rows', 'columns')
print('img named:', img_named.shape, img_named.names)
print('batch named:', batch_named.shape, batch_named.names)

img named: torch.Size([3, 5, 5]) ('channels', 'rows', 'columns')
batch named: torch.Size([2, 3, 5, 5]) (None, 'channels', 'rows', 'columns')


对于2个输入的操作，PyTorch除了进行常规的维度检查（维度是否相同），还将检查张量的名称。
目前还没有自动维度对齐的功能，因此需要**显式地进行操作**。

**align_as()方法将返回一个张量，其中添加了缺失的维度！**

In [29]:
weights_aligned = weights_named.align_as(img_named)
weights_aligned.shape, weights_aligned.names

(torch.Size([3, 1, 1]), ('channels', 'rows', 'columns'))

接收维度参数的函数（如sum()），**同样也接收命名维度**

In [30]:
gray_named = (img_named * weights_aligned).sum('channels')
gray_named.shape, gray_named.names

(torch.Size([5, 5]), ('rows', 'columns'))

如果尝试将不同名称的维度组合起来，会出现错误。

In [31]:
# gray_named = (img_named[..., :3] * weights_named).sum('channels')

如果我们想在 对命名的张量进行操作的函数之外 使用张量，**需要重命名张量名称为None来删除名称**。

In [32]:
gray_plain = gray_named.rename(None)
gray_plain.shape, gray_plain.names

(torch.Size([5, 5]), (None, None))

## 3.5 张量的元素类型

### 3.5.1 使用dtype指定数字类型

张量构造函数通过dtype参数指定包含在张量中的数字数据类型，如**tensor()、zeros()、ones()**

torch.float32或torch.float

f64、double

int8、uint8

...

int64、long

torch.bool

### 3.5.3管理张量的dtype属性

In [33]:
double_points = torch.ones(10, 2, dtype=torch.double)
short_points = torch.tensor([[1, 2], [3, 4]], dtype=torch.short)

通过访问相应的**属性**来了解张量的dtype值。

In [34]:
short_points.dtype

torch.int16

我们还可以使用相应的**转换方法**将张量创建函数的输出转换为正确的类型

In [35]:
double_points = torch.zeros(10, 2).double()
short_points = torch.ones(10, 2).short()
double_points.dtype, short_points.dtype

(torch.float64, torch.int16)

或者使用更为方便的方法**to()**

In [36]:
double_points = torch.zeros(10, 2).to(torch.double)
short_points = torch.ones(10, 2).to(torch.short)
double_points.dtype, short_points.dtype

(torch.float64, torch.int16)

在操作中使用多种类型时，输入会自动向较大类型进行转换。

## 3.6 张量的api

首先，关于张量与张量之间的绝大多数操作（张量对象的方法）都可以再torch模块中找到，如transpose()函数。

torch.transpose(input, dim0, dim1)

参数说明:

input (Tensor): 要进行操作的输入张量。

dim0 (int): 要交换的第一个维度。

dim1 (int): 要交换的第二个维度。

函数会返回一个新的张量，该张量是输入张量在指定维度上进行交换后的结果。重要的是，返回的张量与输入张量共享底层数据存储，这意味着对其中一个张量的修改会影响到另一个。这使得 transpose 操作非常高效，因为它避免了不必要的数据复制。如果你需要一个独立的数据副本，可以在结果上调用 .clone() 或 .contiguous().clone() 方法。

In [37]:
a = torch.ones(3, 2)
a_t = torch.transpose(a, 0, 1)
a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))

或者transpose也可以作为张量的一个方法。

In [38]:
a = torch.ones(3, 2)
a_t = a.transpose(0, 1)
a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))